# Supplementary Table S2 -> Plot: Granularity of energy and efficiency reporting

This notebook converts Supplementary Table S2 into a compact heatmap-style plot.
Values: `No`=0, `Partial`/`Limited`=1, `Yes`=2.
Default: `SAVE = False` (do not write files). Set `SAVE = True` to export a PNG to `plots_for_paper/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

WORKBOOK_NAME = "disclosure_coding.csv"

def find_plot_project():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "plots_for_paper"]
    for candidate in candidates:
        if (candidate / "data" / WORKBOOK_NAME).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate data/{WORKBOOK_NAME}")

PROJECT_DIR = find_plot_project()
DATA_FILE = PROJECT_DIR / "data" / WORKBOOK_NAME
PLOTS_DIR = PROJECT_DIR / "plots"
plot_data = pd.read_csv(DATA_FILE).sort_values("Plot order")

# Set Nature-style matplotlib parameters
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['xtick.major.size'] = 2
plt.rcParams['ytick.major.size'] = 2

SAVE = True

# Supplementary Table S2 is derived directly from Plot_Data.
source_columns = {
    'Facility-level': 'Per-datacenter\nelectricity',
    'Region-resolved': 'Per-region\nelectricity',
    'DC-only MWh': 'Global DC\nelectricity',
    'PUE': 'PUE'
}
status_codes = {"No": 0, "Partial": 1, "Yes": 2}
value_labels = {value: key for key, value in status_codes.items()}

df_status = plot_data.set_index('Company')[list(source_columns)].rename(columns=source_columns)
unknown = set(df_status.stack().dropna().unique()) - set(status_codes)
if unknown:
    raise ValueError(f"Unknown disclosure status values in Plot_Data: {sorted(unknown)}")
df = df_status.replace(status_codes).astype(int)

n_rows, n_cols = df.shape
colors_nature = {
    0: '#E0E0E0',
    1: '#90CAF9',
    2: '#2E7D32'
}
legend_elements = [
    mpatches.Patch(facecolor=colors_nature[2], edgecolor='#1B5E20', label='Yes'),
    mpatches.Patch(facecolor=colors_nature[1], edgecolor='#42A5F5', label='Partial'),
    mpatches.Patch(facecolor=colors_nature[0], edgecolor='#BDBDBD', label='No')
]

fig, ax = plt.subplots(figsize=(5, 3), dpi=300)
fig.patch.set_facecolor('white')
for i in range(n_rows):
    for j in range(n_cols):
        value = df.iloc[i, j]
        rect = mpatches.Rectangle(
            (j, i), 1, 1, facecolor=colors_nature[value],
            edgecolor='#999999', linewidth=0.8, zorder=2
        )
        ax.add_patch(rect)
        ax.text(
            j + 0.5, i + 0.5, value_labels[value],
            ha='center', va='center', fontsize=7,
            color='white' if value == 2 else '#444444'
        )

ax.set_xlim(0, n_cols)
ax.set_ylim(0, n_rows)
ax.invert_yaxis()
ax.set_xticks(np.arange(n_cols) + 0.5)
ax.set_yticks(np.arange(n_rows) + 0.5)
ax.set_xticklabels(df.columns, fontsize=8, ha='center')
ax.set_yticklabels(df.index, fontsize=8)
ax.tick_params(length=0, pad=3)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(
    handles=legend_elements, loc='upper left', bbox_to_anchor=(1.01, 1),
    frameon=False, fontsize=8, handlelength=1.2, handleheight=1.2
)
plt.tight_layout()

if SAVE:
    out_pdf = PLOTS_DIR / 'granularity_S2.pdf'
    out_png = PLOTS_DIR / 'granularity_S2.png'
    fig.savefig(out_pdf, dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'Saved {out_pdf} and {out_png}')

plt.show()


**Notes**:
- Edit the `data/disclosure_coding.csv` table to change source classifications.
- Re-run this notebook to regenerate the PDF and PNG in `plots/`.
